# 00 · Config & helpers (Codespaces · Turso → DuckDB medallion)
Same medallion pattern as the Databricks version, rebuilt for a GitHub Codespace:

| Databricks version | This version |
|---|---|
| S3 landing + Auto Loader | **Turso** source + incremental extract (`updated_at` watermark) → Parquet landing files |
| Delta tables on Unity Catalog | **DuckDB** file (`lake/warehouse.duckdb`) with `bronze` / `silver` / `gold` schemas |
| Delta Change Data Feed | `_silver_batch` column on Silver (which run last changed the row) |
| Delta `MERGE` | DuckDB `MERGE INTO` (DuckDB 1.4+) |

Run from the next notebooks with `%run -i ./00_config.ipynb`. Adding a source = one entry in `ENTITIES`.

In [1]:
import os
from pathlib import Path
import duckdb, pandas as pd

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT          # works from repo root or notebooks/
LAKE = ROOT / "lake"; LANDING = LAKE / "landing"; LAKE.mkdir(exist_ok=True)

LOAD_MODE = os.getenv("LOAD_MODE", "incremental")   # Gold load: 'incremental' (default) or 'full'
CHUNK     = 50_000                                  # rows per Parquet part: bounds memory at any volume

## Source connection (Turso)
Uses Turso when `TURSO_DATABASE_URL` and `TURSO_AUTH_TOKEN` are set (Codespaces secrets), otherwise a local SQLite file so you can try everything offline. Default is an **embedded replica** (a local copy kept in sync, so extracts are fast); set `TURSO_REMOTE=1` to query the cloud database directly.

In [2]:
import libsql                                       # pip install libsql  (Turso's Python SDK)

TURSO_URL, TURSO_TOKEN = os.getenv("TURSO_URL"), os.getenv("TURSO_TOKEN")

def source_conn():
    if TURSO_URL and TURSO_TOKEN:
        if os.getenv("TURSO_REMOTE") == "1":
            return libsql.connect(database=TURSO_URL, auth_token=TURSO_TOKEN)
        conn = libsql.connect(str(LAKE / "turso_replica.db"), sync_url=TURSO_URL, auth_token=TURSO_TOKEN)
        conn.sync()                                  # pull latest changes into the local replica
        return conn
    return libsql.connect(str(LAKE / "source_local.db"))

def src_rows(conn, sql, params=()):
    cur = conn.cursor(); cur.execute(sql, params)
    cols = [d[0] for d in cur.description]
    while rows := cur.fetchmany(CHUNK):
        yield cols, rows

## Warehouse, control tables & run batch
`ctl.watermark` holds every incremental position (source extract time per entity, last Bronze batch cleaned, last Silver batch loaded to Gold). Watermarks only move **after** a step succeeds, so a failed run is safely re-run.

In [3]:
con = None
BATCH_ID = None

def open_warehouse():
    global con
    con = duckdb.connect(str(LAKE / "warehouse.duckdb"))
    for s in ("bronze", "silver", "gold", "ctl"):
        con.execute(f"CREATE SCHEMA IF NOT EXISTS {s}")
    con.execute("CREATE TABLE IF NOT EXISTS ctl.watermark (name VARCHAR PRIMARY KEY, value VARCHAR, updated_ts TIMESTAMP)")
    con.execute("CREATE TABLE IF NOT EXISTS ctl.batch_log (batch_id BIGINT PRIMARY KEY, started_ts TIMESTAMP, load_mode VARCHAR)")
    con.execute("CREATE TABLE IF NOT EXISTS silver._quarantine (entity VARCHAR, batch_id BIGINT, payload JSON, _ts TIMESTAMP)")
    return con

def get_wm(name):
    r = con.execute("SELECT value FROM ctl.watermark WHERE name = ?", [name]).fetchone()
    return r[0] if r else None

def set_wm(name, value):
    con.execute("""INSERT INTO ctl.watermark VALUES (?, ?, now())
                   ON CONFLICT (name) DO UPDATE SET value = excluded.value, updated_ts = excluded.updated_ts""", [name, str(value)])

def new_batch():
    global BATCH_ID
    BATCH_ID = con.execute("SELECT coalesce(max(batch_id), 0) + 1 FROM ctl.batch_log").fetchone()[0]
    con.execute("INSERT INTO ctl.batch_log VALUES (?, now(), ?)", [BATCH_ID, LOAD_MODE])
    return BATCH_ID

## Metadata: the only thing that changes per source
Column tuples are `(name, DuckDB type[, text transform])`. Bronze keeps everything as text; Silver applies these types, transforms and rules.

In [4]:
ENTITIES = {
    "customers": dict(keys=["customer_id"], order_by="updated_at",
        columns=[("customer_id", "BIGINT"), ("name", "VARCHAR"), ("email", "VARCHAR", "lower"),
                 ("segment", "VARCHAR", "lower"), ("country", "VARCHAR", "upper"), ("updated_at", "TIMESTAMP")],
        required=["customer_id", "updated_at"], rules=[]),
    "products": dict(keys=["product_id"], order_by="updated_at",
        columns=[("product_id", "BIGINT"), ("product_name", "VARCHAR"), ("category", "VARCHAR", "lower"),
                 ("list_price", "DECIMAL(12,2)"), ("updated_at", "TIMESTAMP")],
        required=["product_id", "updated_at"], rules=["list_price >= 0"]),
    "sales": dict(keys=["order_line_id"], order_by="updated_at",
        columns=[("order_line_id", "BIGINT"), ("order_id", "BIGINT"), ("customer_id", "BIGINT"), ("product_id", "BIGINT"),
                 ("quantity", "INTEGER"), ("unit_price", "DECIMAL(12,2)"), ("order_ts", "TIMESTAMP"), ("updated_at", "TIMESTAMP")],
        required=["order_line_id", "order_id", "order_ts", "updated_at"], rules=["quantity > 0", "unit_price >= 0"]),
}
DIMS = {   # Gold dimensions (SCD Type 1); surrogate keys come from a sequence, -1 = Unknown member
    "dim_customer": dict(src="customers", key="customer_id", sk="customer_sk", attrs=["name", "email", "segment", "country"]),
    "dim_product":  dict(src="products",  key="product_id",  sk="product_sk",  attrs=["product_name", "category", "list_price"]),
}

## Step 1 helper · Stage: Turso → Parquet landing → Bronze
Extracts only rows with `updated_at >= last watermark`, writes immutable Parquet files (replayable, like an S3 landing zone), then appends to Bronze as text plus lineage columns (`_batch_id`, `_ingest_ts`, `_source_file`). Rows already in Bronze (same key and `updated_at`) are skipped, so re-runs never duplicate. New source columns are added to Bronze automatically.

In [5]:
def stage(entity):
    key = ENTITIES[entity]["keys"][0]
    wm = get_wm(f"extract.{entity}")                              # ISO timestamps compare correctly as text
    sql = f"SELECT * FROM {entity}" + (" WHERE updated_at >= ?" if wm else "") + f" ORDER BY updated_at, {key}"
    out = LANDING / entity / f"batch_id={BATCH_ID}"; out.mkdir(parents=True, exist_ok=True)
    n, max_ts = 0, wm or ""
    for i, (cols, rows) in enumerate(src_rows(src, sql, (wm,) if wm else ())):
        df = pd.DataFrame(rows, columns=cols).astype("string")
        df.to_parquet(out / f"part-{i:05d}.parquet", index=False)
        n += len(df); max_ts = max(max_ts, df["updated_at"].max())
    if n == 0:
        print(f"  {entity:10s} nothing new in source"); return
    pq = f"'{out}/*.parquet', hive_partitioning = false"
    cols = [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM read_parquet({pq})").fetchall()]
    cast = ", ".join(f'CAST(s."{c}" AS VARCHAR) AS "{c}"' for c in cols)
    con.execute(f"""CREATE TABLE IF NOT EXISTS bronze.{entity} AS
                    SELECT {cast.replace('s."', '"')}, CAST(0 AS BIGINT) AS _batch_id,
                           CAST(now() AS TIMESTAMP) AS _ingest_ts, CAST('' AS VARCHAR) AS _source_file
                    FROM read_parquet({pq}) LIMIT 0""")
    have = {r[0] for r in con.execute(f"SELECT column_name FROM information_schema.columns WHERE table_schema='bronze' AND table_name='{entity}'").fetchall()}
    for c in cols:
        if c not in have: con.execute(f'ALTER TABLE bronze.{entity} ADD COLUMN "{c}" VARCHAR')   # schema evolution
    names = ", ".join(f'"{c}"' for c in cols)
    con.execute(f"""INSERT INTO bronze.{entity} ({names}, _batch_id, _ingest_ts, _source_file)
                    SELECT {cast}, {BATCH_ID}, now(), s.filename
                    FROM read_parquet({pq}, filename = true) AS s
                    WHERE NOT EXISTS (SELECT 1 FROM bronze.{entity} b WHERE b."{key}" = s."{key}" AND b.updated_at = s.updated_at)""")
    new = con.execute(f"SELECT count(*) FROM bronze.{entity} WHERE _batch_id = {BATCH_ID}").fetchone()[0]
    set_wm(f"extract.{entity}", max_ts)                           # only after Bronze succeeded
    print(f"  {entity:10s} extracted {n:5d} rows (since {wm or 'the beginning'}), {new:5d} new in bronze")

## Step 2 helpers · Clean: Bronze → Silver
`try_cast` to real types, trim / nullify blanks / lower / upper, DQ rules (failures go to `silver._quarantine` as JSON of the **raw** row), latest record per key, then a guarded `MERGE`. Only rows whose values actually changed are updated, and those get `_silver_batch = this run`; that column is what makes the Gold load incremental.

In [6]:
def merge(target, source, keys, cols, batch_col, guard="true", sk=None):
    """Generic upsert: insert new keys, update only when a value changed (and the guard passes)."""
    attrs = [c for c in cols if c not in keys]
    on    = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    diff  = " OR ".join(f"t.{c} IS DISTINCT FROM s.{c}" for c in attrs)
    sets  = ", ".join(f"{c} = s.{c}" for c in attrs) + f", {batch_col} = {BATCH_ID}"
    icols = ([sk[0]] if sk else []) + cols + [batch_col]
    ivals = ([sk[1]] if sk else []) + [f"s.{c}" for c in cols] + [str(BATCH_ID)]
    return con.execute(f"""MERGE INTO {target} AS t USING {source} AS s ON {on}
        WHEN MATCHED AND ({guard}) AND ({diff}) THEN UPDATE SET {sets}
        WHEN NOT MATCHED THEN INSERT ({', '.join(icols)}) VALUES ({', '.join(ivals)})""").fetchone()[0]

def _expr(col):
    name, typ, *fn = col
    base = f"nullif(trim(\"{name}\"), '')"
    if typ.startswith("VARCHAR"):
        return f"{fn[0]}({base})" if fn else base
    return f"try_cast({base} AS {typ})"

def clean_to_silver(entity):
    c, cols = ENTITIES[entity], ENTITIES[entity]["columns"]
    names, keys = [x[0] for x in cols], ENTITIES[entity]["keys"]
    last = int(get_wm(f"silver.{entity}") or 0)
    top = con.execute(f"SELECT coalesce(max(_batch_id), 0) FROM bronze.{entity}").fetchone()[0]
    if top <= last:
        print(f"  {entity:10s} nothing new in bronze"); return
    checks = [f"{k} IS NOT NULL" for k in c["required"]] + c["rules"]
    valid  = "coalesce(" + " AND ".join(f"({x})" for x in checks) + ", false)"
    con.execute(f"""CREATE OR REPLACE TEMP TABLE _typed AS
        SELECT {', '.join(f'{_expr(x)} AS "{x[0]}"' for x in cols)}, rowid AS _rid, _ingest_ts
        FROM bronze.{entity} WHERE _batch_id > {last}""")
    con.execute(f"CREATE OR REPLACE TEMP TABLE _flagged AS SELECT *, {valid} AS _valid FROM _typed")
    bad = con.execute(f"""INSERT INTO silver._quarantine
        SELECT '{entity}', {BATCH_ID}, to_json(b), now() FROM bronze.{entity} b
        JOIN _flagged f ON b.rowid = f._rid WHERE NOT f._valid""").fetchone()[0]
    con.execute(f"""CREATE OR REPLACE TEMP TABLE _good AS
        SELECT {', '.join(names)} FROM _flagged WHERE _valid
        QUALIFY row_number() OVER (PARTITION BY {', '.join(keys)} ORDER BY {c['order_by']} DESC, _ingest_ts DESC, _rid DESC) = 1""")
    con.execute(f"""CREATE TABLE IF NOT EXISTS silver.{entity}
        ({', '.join(f'{x[0]} {x[1]}' for x in cols)}, _silver_batch BIGINT, PRIMARY KEY ({', '.join(keys)}))""")
    ob = c["order_by"]
    changed = merge(f"silver.{entity}", "_good", keys, names, "_silver_batch", guard=f"s.{ob} >= t.{ob}")
    set_wm(f"silver.{entity}", top)
    print(f"  {entity:10s} {con.execute('SELECT count(*) FROM _good').fetchone()[0]:5d} valid latest rows -> {changed:5d} inserted/updated, {bad:3d} quarantined")

## Steps 3–5 helpers · Join, merge to Gold, and the incremental delta
* **Delta (step 5):** `delta_predicate` picks the Silver rows changed since the last Gold load (`_silver_batch > watermark`), or everything in `full` mode and on the first run. The watermark is recorded only after the Gold merge succeeds.
* **Join (step 3):** the Silver sales delta is left-joined to the Gold dimensions; a missing dimension row resolves to the Unknown member (`-1`). In incremental mode, facts that previously landed with an unknown key are re-driven so they self-heal when the dimension arrives.
* **Merge (step 4):** idempotent upsert into the Gold dimensions and fact.

In [7]:
PENDING = {}

def delta_predicate(target, silver_table):
    wm  = get_wm(f"gold.{target}")
    end = con.execute(f"SELECT coalesce(max(_silver_batch), 0) FROM silver.{silver_table}").fetchone()[0]
    PENDING[target] = end
    if LOAD_MODE == "full" or wm is None:
        return "true"
    return f"s._silver_batch > {int(wm)} AND s._silver_batch <= {end}"

def load_dim(dim):
    d = DIMS[dim]; tgt, seq = f"gold.{dim}", f"gold.seq_{DIMS[dim]['sk']}"
    types = {x[0]: x[1] for x in ENTITIES[d["src"]]["columns"]}
    con.execute(f"CREATE SEQUENCE IF NOT EXISTS {seq} START 1")
    con.execute(f"""CREATE TABLE IF NOT EXISTS {tgt} ({d['sk']} BIGINT PRIMARY KEY, {d['key']} BIGINT UNIQUE NOT NULL,
                    {', '.join(f'{a} {types[a]}' for a in d['attrs'])}, _gold_batch BIGINT)""")
    con.execute(f"INSERT INTO {tgt} ({d['sk']}, {d['key']}) SELECT -1, -1 WHERE NOT EXISTS (SELECT 1 FROM {tgt} WHERE {d['sk']} = -1)")
    pred, cols = delta_predicate(dim, d["src"]), [d["key"], *d["attrs"]]
    changed = merge(tgt, f"(SELECT {', '.join(cols)} FROM silver.{d['src']} s WHERE {pred})", [d["key"]], cols,
                    "_gold_batch", sk=(d["sk"], f"nextval('{seq}')"))
    set_wm(f"gold.{dim}", PENDING[dim])
    print(f"  {dim:13s} {changed:5d} inserted/updated")

FACT = "gold.fact_sales"
FACT_COLS = ["order_line_id", "order_id", "order_date", "order_ts", "customer_sk", "product_sk",
             "quantity", "unit_price", "amount"]

def build_fact_join():
    con.execute(f"""CREATE TABLE IF NOT EXISTS {FACT} (order_line_id BIGINT PRIMARY KEY, order_id BIGINT, order_date DATE,
        order_ts TIMESTAMP, customer_sk BIGINT, product_sk BIGINT, quantity INTEGER, unit_price DECIMAL(12,2),
        amount DECIMAL(14,2), _gold_batch BIGINT)""")
    pred = delta_predicate("fact_sales", "sales")
    late = ""
    if pred != "true":                                           # incremental: also re-drive facts with an unknown key
        late = f" OR s.order_line_id IN (SELECT order_line_id FROM {FACT} WHERE customer_sk = -1 OR product_sk = -1)"
    con.execute(f"""CREATE OR REPLACE TEMP TABLE _fact AS
        SELECT s.order_line_id, s.order_id, CAST(s.order_ts AS DATE) AS order_date, s.order_ts,
               coalesce(c.customer_sk, -1) AS customer_sk, coalesce(p.product_sk, -1) AS product_sk,
               s.quantity, s.unit_price, CAST(s.quantity * s.unit_price AS DECIMAL(14,2)) AS amount
        FROM silver.sales s
        LEFT JOIN gold.dim_customer c ON c.customer_id = s.customer_id
        LEFT JOIN gold.dim_product  p ON p.product_id  = s.product_id
        WHERE ({pred}){late}""")
    return con.execute("SELECT count(*) FROM _fact").fetchone()[0]

def merge_fact():
    changed = merge(FACT, "_fact", ["order_line_id"], FACT_COLS, "_gold_batch")
    set_wm("gold.fact_sales", PENDING["fact_sales"])
    return changed